# Optimization Algorithms in Operations Research
This notebook provides highly robust, edge-case tested implementations of:
1. **The Big-M Simplex Method** (Linear Programming)
2. **Transportation Problem Solvers** (VAM + MODI)

Each implementation is designed to handle degeneracies, alternative optimums, infeasibility, and unboundedness seamlessly.


## 1. The Big-M Simplex Method
This solver mathematically handles $\le$, $\ge$, and $=$ constraints.

**Robustness Features Added:**
* **Negative RHS Handling:** Automatically normalizes constraints like $2x \le -5$ to $-2x \ge 5$ before starting.
* **Bland's Rule:** Strictly breaks ties during pivot operations using the smallest index, mathematically preventing infinite cycling loops.
* **Complete Outcome Detection:** Intelligently returns `Optimal`, `Multiple Optimal Solutions`, `Infeasible`, or `Unbounded` status flags by analyzing the final tableau structure and Artificial Variables.


In [6]:
import numpy as np

class BigMSimplex:
    def __init__(self, obj_coeffs, constraints, rhs, signs, mode='max', M=1e6, tol=1e-8):
        self.mode = mode.lower()
        self.M = M
        self.tol = tol
        self.num_vars = len(obj_coeffs)
        self.num_cons = len(constraints)
        
        self.c = np.array(obj_coeffs, dtype=float)
        if self.mode == 'min': self.c = -self.c
            
        self.A = np.array(constraints, dtype=float)
        self.b = np.array(rhs, dtype=float)
        self.signs = list(signs)
        
        # Enforce RHS >= 0
        for i in range(self.num_cons):
            if self.b[i] < 0:
                self.b[i] *= -1
                self.A[i] *= -1
                self.signs[i] = '>=' if self.signs[i] == '<=' else '<=' if self.signs[i] == '>=' else '='

        self.slacks = self.surplus = self.artificials = 0
        self.basic_vars = [] 
        self.var_names = [f"x{i+1}" for i in range(self.num_vars)]
        self._setup_tableau()

    def _setup_tableau(self):
        for sign in self.signs:
            if sign == '<=': self.slacks += 1
            elif sign == '>=': self.surplus += 1; self.artificials += 1
            elif sign == '=': self.artificials += 1
                
        total_vars = self.num_vars + self.slacks + self.surplus + self.artificials
        self.tableau = np.zeros((self.num_cons + 1, total_vars + 1))
        self.tableau[:-1, 0:self.num_vars] = self.A
        self.tableau[:-1, -1] = self.b
        self.tableau[-1, 0:self.num_vars] = -self.c
        
        col_idx = self.num_vars
        s_idx, sur_idx, a_idx = 1, 1, 1
        
        for i, sign in enumerate(self.signs):
            if sign == '<=':
                self.tableau[i, col_idx] = 1
                self.basic_vars.append(col_idx)
                self.var_names.append(f"s{s_idx}")
                s_idx += 1; col_idx += 1
            elif sign == '>=':
                self.tableau[i, col_idx] = -1  
                self.var_names.append(f"sur{sur_idx}")
                sur_idx += 1; col_idx += 1
                self.tableau[i, col_idx] = 1   
                self.tableau[-1, col_idx] = self.M 
                self.basic_vars.append(col_idx)
                self.var_names.append(f"A{a_idx}")
                a_idx += 1; col_idx += 1
            elif sign == '=':
                self.tableau[i, col_idx] = 1   
                self.tableau[-1, col_idx] = self.M 
                self.basic_vars.append(col_idx)
                self.var_names.append(f"A{a_idx}")
                a_idx += 1; col_idx += 1
                
        # Zero out artificial variable M costs in objective row
        for i, basic_col in enumerate(self.basic_vars):
            if self.tableau[-1, basic_col] != 0:
                self.tableau[-1, :] -= self.tableau[-1, basic_col] * self.tableau[i, :]

    def solve(self):
        iteration = 0
        while True:
            iteration += 1
            if np.all(self.tableau[-1, :-1] >= -self.tol):
                return self._compile_result("Optimal", iteration)
                
            # Entering variable (Bland's Rule: smallest index)
            enter_col = np.where(self.tableau[-1, :-1] < -self.tol)[0][0]
            
            # Leaving variable (Ratio test)
            ratios = np.full(self.num_cons, np.inf)
            for i in range(self.num_cons):
                if self.tableau[i, enter_col] > self.tol:
                    ratios[i] = self.tableau[i, -1] / self.tableau[i, enter_col]
            
            if np.all(ratios == np.inf): return self._compile_result("Unbounded", iteration)
                
            # Bland's Rule tie-breaker
            min_ratio = np.min(ratios)
            min_indices = np.where(ratios == min_ratio)[0]
            leave_row = min_indices[np.argmin([self.basic_vars[i] for i in min_indices])]
            
            # Pivot operations
            self.tableau[leave_row, :] /= self.tableau[leave_row, enter_col]
            for i in range(self.num_cons + 1):
                if i != leave_row:
                    self.tableau[i, :] -= self.tableau[i, enter_col] * self.tableau[leave_row, :]
            self.basic_vars[leave_row] = enter_col

    def _compile_result(self, status, iterations):
        solution = np.zeros(len(self.var_names))
        for i, basic_col in enumerate(self.basic_vars):
            solution[basic_col] = self.tableau[i, -1]
            
        final_z = -self.tableau[-1, -1] if self.mode == 'min' else self.tableau[-1, -1]
        
        if status == "Optimal":
            # Infeasible if Artificial > 0
            if any(self.var_names[i].startswith('A') and solution[i] > self.tol for i in range(len(self.var_names))):
                status = "Infeasible"
            else:
                # Alternative optimum check
                alt_opt = False
                for col in range(len(self.var_names)):
                    if col not in self.basic_vars and not self.var_names[col].startswith('A'):
                        if abs(self.tableau[-1, col]) <= self.tol:
                            alt_opt = True
                            break
                            
                # Degeneracy check
                is_degenerate = any(abs(solution[bc]) <= self.tol for bc in self.basic_vars)
                
                if alt_opt:
                    status = "Optimal (Alternative solutions exist)"
                elif is_degenerate:
                    status = "Optimal (Degenerate)"
                    
        if status == "Infeasible":
            return {
                "status": status,
                "objective_value": None,
                "variables": None,
                "iterations": iterations
            }
        elif status == "Unbounded":
            return {
                "status": status,
                "objective_value": float('inf'),
                "variables": None,
                "iterations": iterations
            }
            
        decision_vars = [round(solution[i], 4) for i in range(self.num_vars)]
        return {
            "status": status,
            "objective_value": round(final_z, 4),
            "variables": decision_vars,
            "iterations": iterations
        }


### Example Usage: Big-M Simplex
Test Cases for Big-M Method.


In [7]:
# Test Case 1
print("--- Test Case 1 ---")
solver_1 = BigMSimplex(
    obj_coeffs=[2, 3],
    constraints=[
        [1, 1],
        [1, 2],
        [1, 3]
    ],
    rhs=[5, 6, 12],
    signs=['>=', '>=', '<='],
    mode='min'
)
result = solver_1.solve()
print("Status:", result["status"])
print("Objective Value:", result["objective_value"])
print("Variables:", result["variables"])

# Test Case 2
print("\n--- Test Case 2 ---")
solver_2 = BigMSimplex(
    obj_coeffs=[3, 2],
    constraints=[
        [1, 1],
        [1, 1]
    ],
    rhs=[2, 4],
    signs=['<=', '>='],
    mode='max'
)
result = solver_2.solve()
print("Status:", result["status"])
print("Objective Value:", result["objective_value"])
print("Variables:", result["variables"])

# Test Case 4
print("\n--- Test Case 4 ---")
solver_4 = BigMSimplex(
    obj_coeffs=[4, 6],
    constraints=[
        [2, 3],
        [1, 1]
    ],
    rhs=[12, 2],
    signs=['<=', '>='],
    mode='max'
)
result = solver_4.solve()
print("Status:", result["status"])
print("Objective Value:", result["objective_value"])
print("Variables:", result["variables"])
# Test Case 3
print("\n--- Test Case 3 ---")
solver_3 = BigMSimplex(
    obj_coeffs=[2, 3],
    constraints=[
        [1, 1],
        [-1, 1]
    ],
    rhs=[2, 4],
    signs=['>=', '<='],
    mode='max'
)
result = solver_3.solve()
print("Status:", result["status"])
print("Objective Value:", result["objective_value"])
print("Variables:", result["variables"])
# Test Case 5
print("\n--- Test Case 5 ---")
solver_5 = BigMSimplex(
    obj_coeffs=[3, 9],
    constraints=[
        [1, 4],
        [1, 2]
    ],
    rhs=[8, 4],
    signs=['<=', '>='],
    mode='max'
)
result = solver_5.solve()
print("Status:", result["status"])
print("Objective Value:", result["objective_value"])
print("Variables:", result["variables"])

--- Test Case 1 ---
Status: Optimal
Objective Value: 11.0
Variables: [np.float64(4.0), np.float64(1.0)]

--- Test Case 2 ---
Status: Infeasible
Objective Value: None
Variables: None

--- Test Case 4 ---
Status: Optimal (Alternative solutions exist)
Objective Value: 24.0
Variables: [np.float64(0.0), np.float64(4.0)]

--- Test Case 3 ---
Status: Unbounded
Objective Value: inf
Variables: None

--- Test Case 5 ---
Status: Optimal
Objective Value: 24.0
Variables: [np.float64(8.0), np.float64(0.0)]


## 2. Transportation Problem
Divided into two distinct phases for structural safety: **VAM** (Initialization) and **MODI** (Optimization).

**Robustness Features Added:**
* **Simultaneous Degeneracy:** VAM handles instances where Supply and Demand deplete exactly at the same time by intentionally leaving a zero-demand column active, forcing an explicit $\epsilon$ zero-allocation to maintain $m+n-1$ basic cells.
* **Pruning Loop Search:** Replaced recursive DFS with a deterministic "pruning" algorithm for Stepping Stone loop discovery, resolving crashing loops in highly degenerate matrices.
* **Balancing & Edge Cases:** Dynamically manages dummy rows/cols and successfully detects Multiple Optimal paths in transportation nodes.


In [8]:
class VAMSolver:
    def __init__(self, costs, supply, demand):
        self.costs = np.array(costs, dtype=float)
        self.supply = np.array(supply, dtype=float)
        self.demand = np.array(demand, dtype=float)
        self.num_rows, self.num_cols = len(supply), len(demand)
        self._balance_problem()
        
    def _balance_problem(self):
        total_sup, total_dem = np.sum(self.supply), np.sum(self.demand)
        if total_sup > total_dem:
            self.demand = np.append(self.demand, total_sup - total_dem)
            self.costs = np.column_stack((self.costs, np.zeros(self.num_rows)))
            self.num_cols += 1
            print("INFO: Added dummy destination.")
        elif total_dem > total_sup:
            self.supply = np.append(self.supply, total_dem - total_sup)
            self.costs = np.vstack((self.costs, np.zeros(self.num_cols)))
            self.num_rows += 1
            print("INFO: Added dummy source.")

    def solve(self):
        sup_c, dem_c, costs_c = self.supply.copy(), self.demand.copy(), self.costs.copy()
        allocation = np.zeros((self.num_rows, self.num_cols))
        basic_cells = []
        active_rows, active_cols = list(range(self.num_rows)), list(range(self.num_cols))

        while active_rows and active_cols:
            row_pen = []
            for i in active_rows:
                v = np.sort(costs_c[i, active_cols])
                pen = v[1] - v[0] if len(v) > 1 else v[0]
                row_pen.append((pen, i, v[0])) # penalty, row_idx, lowest_cost
                
            col_pen = []
            for j in active_cols:
                v = np.sort(costs_c[active_rows, j])
                pen = v[1] - v[0] if len(v) > 1 else v[0]
                col_pen.append((pen, j, v[0]))
                
            # Tie breaking: Max penalty -> Min cost
            max_r = max(row_pen, key=lambda x: (x[0], -x[2]))
            max_c = max(col_pen, key=lambda x: (x[0], -x[2]))
            
            if (max_r[0], -max_r[2]) >= (max_c[0], -max_c[2]):
                r_idx = max_r[1]
                c_idx = active_cols[np.argmin(costs_c[r_idx, active_cols])]
            else:
                c_idx = max_c[1]
                r_idx = active_rows[np.argmin(costs_c[active_rows, c_idx])]
                
            qty = min(sup_c[r_idx], dem_c[c_idx])
            allocation[r_idx, c_idx] = qty
            basic_cells.append((r_idx, c_idx))
            sup_c[r_idx] -= qty
            dem_c[c_idx] -= qty
            
            # Critical Degeneracy Protection (Spanning Tree Preserved)
            if sup_c[r_idx] == 0 and dem_c[c_idx] == 0:
                if len(active_rows) > 1:
                    active_rows.remove(r_idx)
                elif len(active_cols) > 1:
                    active_cols.remove(c_idx)
                else:
                    active_rows.remove(r_idx)
                    active_cols.remove(c_idx)
            elif sup_c[r_idx] == 0:
                active_rows.remove(r_idx)
            else:
                active_cols.remove(c_idx)
                
        # Fill remaining required basics (extreme degeneracy backup)
        req_basic = self.num_rows + self.num_cols - 1
        for r in range(self.num_rows):
            for c in range(self.num_cols):
                if len(basic_cells) >= req_basic: break
                if (r, c) not in basic_cells: basic_cells.append((r, c))
                    
        return allocation, basic_cells, self.costs



In [9]:
class MODISolver:
    def __init__(self, costs, initial_allocation, basic_cells):
        self.costs = np.array(costs)
        self.allocation = np.array(initial_allocation)
        self.basic_cells = list(basic_cells)
        self.num_rows, self.num_cols = self.costs.shape
        self.tol = 1e-5

    def _get_loop(self, start_r, start_c):
        cells = self.basic_cells + [(start_r, start_c)]
        while True:
            r_counts = {r: sum(1 for x, y in cells if x == r) for r in range(self.num_rows)}
            c_counts = {c: sum(1 for x, y in cells if y == c) for c in range(self.num_cols)}
            to_remove = [(r, c) for r, c in cells if r_counts[r] == 1 or c_counts[c] == 1]
            if not to_remove: break
            for cell in to_remove: cells.remove(cell)
                
        # Assemble loop ordered
        if not cells: return []
        loop = [(start_r, start_c)]
        cells.remove((start_r, start_c))
        is_row_move = True
        
        while cells:
            curr_r, curr_c = loop[-1]
            next_cell = next(((r, c) for r, c in cells if (is_row_move and r == curr_r) or (not is_row_move and c == curr_c)), None)
            if next_cell:
                loop.append(next_cell)
                cells.remove(next_cell)
                is_row_move = not is_row_move
            else: break
        return loop

    def solve(self):
        iteration = 0
        while True:
            iteration += 1
            u, v = {r: None for r in range(self.num_rows)}, {c: None for c in range(self.num_cols)}
            u[0] = 0 
            
            changed = True
            while changed:
                changed = False
                for r, c in self.basic_cells:
                    if u[r] is not None and v[c] is None:
                        v[c] = self.costs[r, c] - u[r]; changed = True
                    elif v[c] is not None and u[r] is None:
                        u[r] = self.costs[r, c] - v[c]; changed = True
            
            for i in range(self.num_rows):
                if u[i] is None: u[i] = 0
            for j in range(self.num_cols):
                if v[j] is None: v[j] = 0
                
            min_pen, enter_r, enter_c = 0, -1, -1
            for r in range(self.num_rows):
                for c in range(self.num_cols):
                    if (r, c) not in self.basic_cells:
                        pen = self.costs[r, c] - (u[r] + v[c])
                        if pen < min_pen - self.tol:
                            min_pen, enter_r, enter_c = pen, r, c
                            
            if min_pen >= -self.tol:
                status = "Optimal"
                # Check alternative optimal
                for r in range(self.num_rows):
                    for c in range(self.num_cols):
                        if (r, c) not in self.basic_cells and abs(self.costs[r, c] - (u[r] + v[c])) <= self.tol:
                            status = "Multiple Optimal Solutions"
                return {
                    "status": status,
                    "allocation": self.allocation,
                    "cost": np.sum(self.allocation * self.costs),
                    "iterations": iteration
                }
                
            loop = self._get_loop(enter_r, enter_c)
            sub_cells = [loop[i] for i in range(1, len(loop), 2)]
            add_cells = [loop[i] for i in range(0, len(loop), 2)]
            
            theta_cell = min(sub_cells, key=lambda x: self.allocation[x[0], x[1]])
            theta = self.allocation[theta_cell[0], theta_cell[1]]
            
            for r, c in sub_cells: self.allocation[r, c] -= theta
            for r, c in add_cells: self.allocation[r, c] += theta
                
            self.basic_cells.remove(theta_cell)
            self.basic_cells.append((enter_r, enter_c))


### Example Usage: VAM + MODI


In [10]:
costs = [
    [3, 1, 7, 4],
    [2, 6, 5, 9],
    [8, 3, 3, 2]
]
supply = [300, 400, 500]
demand = [250, 350, 400, 200]

print("--- Running VAM ---")
vam = VAMSolver(costs, supply, demand)
alloc, basic, padded_costs = vam.solve()
print("VAM Initial Cost:", np.sum(alloc * padded_costs))

print("\n--- Running MODI ---")
modi = MODISolver(padded_costs, alloc, basic)
result = modi.solve()

print("Status:", result["status"])
print("Final Optimal Cost:", result["cost"])
print("Final Allocation Matrix:\n", result["allocation"])


--- Running VAM ---
VAM Initial Cost: 2850.0

--- Running MODI ---
Status: Optimal
Final Optimal Cost: 2850.0
Final Allocation Matrix:
 [[  0. 300.   0.   0.]
 [250.   0. 150.   0.]
 [  0.  50. 250. 200.]]
